# Building a Classifier

In the previous notebook, we trained a logistic regression model that achieved impressive-looking accuracy on the fraud detection problem. Then we showed that a model which always predicts "not fraud" scores almost as well. Accuracy, it turns out, is the wrong scoreboard for this game.

The explainer we just read introduced the tools we need: the confusion matrix, precision, recall, and the trade-off between false positives and false negatives. We saw that the right balance between these errors is a human judgement — one that depends on the costs of each mistake. Now we put those ideas to work on Northgate Retail Bank's transaction data.

In this notebook we will:

- Use a **confusion matrix** to see exactly where a model succeeds and fails
- Measure **precision**, **recall**, and **F1 score**
- Train a **decision tree** classifier and compare it to logistic regression
- Tune the **classification threshold** to shift the precision/recall balance
- Handle class imbalance with **class weights**

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score,
)

sns.set_style("whitegrid")

---

## 1. Prepare the data

We repeat the setup from the previous notebook: load the transactions, select features, and split into training and test sets. This is the same data preparation pipeline we have already seen.

In [ ]:
df = pd.read_csv("../data/transactions.csv")

features = ["amount", "hour_of_day", "is_international", "distance_from_home_km"]
X = df[features].copy()
X["is_international"] = X["is_international"].astype(int)
y = df["is_fraud"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Fraud in test set: {y_test.sum()} / {len(y_test)}")

---

## 2. Logistic regression baseline

We train the same logistic regression as before so we have a baseline to improve on. The model itself is not the point here; what matters is how we *evaluate* it.

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

---

## 3. The confusion matrix

In the explainer, we learned that every prediction a classifier makes falls into one of four categories. The **confusion matrix** lays all four out in a single table:

| | Predicted negative | Predicted positive |
|---|---|---|
| **Actually negative** | True negatives (TN) | False positives (FP) |
| **Actually positive** | False negatives (FN) | True positives (TP) |

For our fraud problem:
- **True positive (TP)**: fraud correctly detected. The model earned its keep.
- **False positive (FP)**: a legitimate transaction wrongly flagged. The customer gets an unwelcome phone call.
- **False negative (FN)**: fraud that slipped through. The bank absorbs the loss.
- **True negative (TN)**: a legitimate transaction correctly allowed through. The quiet majority.

Let's see what our logistic regression actually did.

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_lr, display_labels=["Legitimate", "Fraud"]).plot(ax=ax, cmap="Blues")
ax.set_title("Logistic Regression")
plt.tight_layout()
plt.show()

Now we can put numbers to what the confusion matrix shows. These three metrics each answer a different question about the model's performance:

- **Precision** = TP / (TP + FP). Of the transactions we flagged as fraud, what proportion actually were? High precision means few false alarms.
- **Recall** = TP / (TP + FN). Of all actual fraud, what proportion did we catch? High recall means little fraud slips through.
- **F1 score** = harmonic mean of precision and recall. A single number that balances both.

The **trade-off** between precision and recall is the central theme of the explainer we read. Flagging more transactions as fraud increases recall (we catch more fraud) but decreases precision (more innocent transactions get blocked). There is no way to maximise both simultaneously.

In [ ]:
print(classification_report(y_test, y_pred_lr, target_names=["Legitimate", "Fraud"]))

The **classification report** shows precision, recall, and F1 for each class. Focus on the Fraud row. If recall is low, the model is missing most fraud cases, which is exactly the problem accuracy was hiding.

---

## 5. Decision tree classifier

Logistic regression is not the only option. A **decision tree** takes a different approach: it learns a series of if/else rules from the data, splitting on whichever feature and threshold best separate the classes at each step. The result is a flowchart-like structure that is easy to interpret.

Let's train one and see whether it does better on our fraud problem.

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

In [ ]:
print("Decision Tree:")
print(classification_report(y_test, y_pred_dt, target_names=["Legitimate", "Fraud"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=["Legit", "Fraud"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Logistic Regression")

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_dt),
                       display_labels=["Legit", "Fraud"]).plot(ax=axes[1], cmap="Oranges")
axes[1].set_title("Decision Tree")

plt.tight_layout()
plt.show()

Compare the confusion matrices side by side. Which model catches more fraud? Which produces more false alarms? There is no single "best" answer without knowing the cost of each type of error to the bank. This is the judgement call the explainer described: the data tells us *what* the model does, but not *which trade-off is acceptable*.

---

## 6. Handling class imbalance with class weights

Both models so far treat every error equally. But the compliance team would argue that missing a fraudulent transaction (false negative) is far worse than briefly inconveniencing a customer with a false alarm (false positive).

Setting `class_weight='balanced'` tells the model to penalise mistakes on the minority class more heavily during training. The model will try harder to catch fraud, at the cost of flagging more legitimate transactions.

In [ ]:
lr_balanced = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
lr_balanced.fit(X_train, y_train)
y_pred_balanced = lr_balanced.predict(X_test)

print("Logistic Regression (balanced class weights):")
print(classification_report(y_test, y_pred_balanced, target_names=["Legitimate", "Fraud"]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_balanced),
    display_labels=["Legit", "Fraud"]
).plot(ax=ax, cmap="Greens")
ax.set_title("Logistic Regression (balanced)")
plt.tight_layout()
plt.show()

Recall should improve significantly — the model now catches more fraud. But look at what happened to precision. More legitimate transactions are being flagged. This is the precision/recall trade-off in action: we gained something and gave something up.

---

## 7. Threshold tuning

So far, we have been accepting the model's default decision boundary. But most classifiers do not just output a class label. They first compute a **probability** of the positive class, then apply a **threshold** (default 0.5) to decide the label.

We can get the probabilities directly and choose our own threshold. This gives us fine-grained control over where on the precision/recall spectrum the model sits.

In [ ]:
y_proba = lr_balanced.predict_proba(X_test)[:, 1]  # probability of fraud

print("Sample probabilities:")
print(y_proba[:10].round(3))

In [ ]:
# Try different thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

print(f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>10}  {'F1':>10}")
print("-" * 46)
for t in thresholds:
    y_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, y_t, zero_division=0)
    r = recall_score(y_test, y_t)
    f = f1_score(y_test, y_t)
    print(f"{t:>10.1f}  {p:>10.3f}  {r:>10.3f}  {f:>10.3f}")

Watch the pattern: lowering the threshold flags more transactions as fraud, which pushes recall up but drags precision down. Raising it does the opposite. There is no universally correct threshold. The right choice depends on the **business cost** of false positives versus false negatives — a decision that belongs to the compliance team, not the algorithm.

---

## 8. ROC curve

The **ROC curve** (Receiver Operating Characteristic) visualises this trade-off across all possible thresholds at once. It plots the true positive rate (recall) against the false positive rate. The **AUC** (area under the curve) summarises overall model quality in a single number: 1.0 is perfect, 0.5 is random guessing.

In [ ]:
# ROC for both models
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]
y_proba_bal = lr_balanced.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_bal, tpr_bal, _ = roc_curve(y_test, y_proba_bal)

auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_bal = roc_auc_score(y_test, y_proba_bal)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC={auc_lr:.3f})")
ax.plot(fpr_bal, tpr_bal, label=f"LR balanced (AUC={auc_bal:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random guess")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()

A higher AUC means the model is better at distinguishing fraud from legitimate across all thresholds. Both models here use the same features and the same underlying logistic regression; the difference is how they weight the classes during training.

---

## 9. Exercises

### Exercise 1: Balanced decision tree

Train a `DecisionTreeClassifier` with `class_weight='balanced'` and `max_depth=5`. Print the classification report and compare recall for the Fraud class against the original (unbalanced) decision tree.

In [ ]:
# Your code here


### Exercise 2: Find the best threshold

Using the balanced logistic regression's predicted probabilities (`y_proba`), loop through thresholds from 0.05 to 0.95 in steps of 0.05. For each, compute the F1 score. Which threshold gives the highest F1? Plot threshold (x-axis) against F1 (y-axis).

In [ ]:
# Your code here


### Exercise 3: Feature importance from the decision tree

Decision trees have a `.feature_importances_` attribute that tells you how much each feature contributed to the splits. Access it from `dt_model`, pair each importance with its feature name, and display them sorted from most to least important. Create a horizontal bar chart.

In [ ]:
# Your code here


### Exercise 4: Business decision

Suppose the bank estimates that each undetected fraud costs £2,000, while each false alarm costs £10 (manual review time). Using the balanced logistic regression at a threshold of 0.3, calculate:

1. The number of false negatives and false positives
2. The total cost of false negatives (count * £2,000)
3. The total cost of false positives (count * £10)
4. The combined cost

Then repeat with a threshold of 0.5. Which threshold is cheaper for the bank?

In [ ]:
# Your code here


---

## Summary

In this notebook we moved beyond accuracy to the metrics that actually matter for imbalanced classification. We:

- Used a **confusion matrix** to break down predictions into TP, FP, TN, FN
- Measured **precision**, **recall**, and **F1 score** and saw why each tells a different story
- Trained a **decision tree** and compared it to logistic regression
- Applied `class_weight='balanced'` to make the model prioritise the minority class
- Adjusted the **classification threshold** using `predict_proba` to control the precision/recall trade-off
- Plotted an **ROC curve** to visualise model discrimination across all thresholds

The thread running through all of this is that classification is not just about getting the right answer — it is about understanding the *types* of mistakes the model makes and deciding which ones the business can tolerate. In the next notebook, we shift from supervised to **unsupervised learning**: clustering customers into segments without any labels to guide us.